# পাঠ ১১ - এজেন্ট-টু-এজেন্ট (A2A) প্রোটোকল


## সেটআপ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A প্রটোকল কী?

এই **Agent-to-Agent (A2A) প্রটোকল** একটি উন্মুক্ত স্ট্যান্ডার্ড যা AI এজেন্টদের যোগাযোগ করতে সক্ষম করে,
একে অপরকে খুঁজে পেতে এবং সহযোগিতা করতে — এমনকি যখন তারা বিভিন্ন ফ্রেমওয়ার্কে নির্মিত বা বিভিন্ন সার্ভিসে হোস্ট করা
বিভিন্ন সার্ভিসে।

মূল ধারণা:

- **অন্বেষণ** – এজেন্টরা তাদের সক্ষমতা বর্ণনা করে এমন একটি *Agent Card* প্রকাশ করে, এটিকে
  অন্যান্য এজেন্টদের (অথবা অর্চেস্ট্রেটরদের) জন্য একটি কাজের জন্য সঠিক বিশেষজ্ঞ খুঁজে পাওয়া সহজ করে তোলে।
- **বার্তা আদানপ্রদান** – এজেন্টরা একটি সাধারণ প্রটোকলের মাধ্যমে কাঠামোবদ্ধ বার্তা বিনিময় করে, ফলে
  একটি এজেন্টের অনুরোধ অন্য একটি এজেন্ট দ্বারা বোঝা এবং পূরণ করা যেতে পারে, তা যার অভ্যন্তরীণ
  বাস্তবায়ন।
- **কাজের জীবনচক্র** – A2A এই রকম অবস্থা নির্ধারণ করে যেমন *submitted*, *working*, *completed*, এবং
  *failed*, অর্চেস্ট্রেটরকে একটি নিয়োগকৃত কাজ কীভাবে অগ্রসর হচ্ছে তার সম্পূর্ণ দৃশ্যমানতা দেয়।

এই পাঠে আমরা A2A-শৈলীর সহযোগিতা অনুকরণ করে তিনটি বিশেষায়িত ট্রাভেল এজেন্টকে
একটি কর্মপ্রবাহে সংযুক্ত করি যেখানে প্রতিটি এজেন্ট তাদের দক্ষতা যোগ করে এবং ফলাফল পরবর্তী এজেন্টকে পৌঁছে দেয়।


## বিশেষায়িত ভ্রমণ এজেন্ট তৈরি করা


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ওয়ার্কফ্লো মাধ্যমে একাধিক এজেন্টের সহযোগিতা

আমরা তিনটি এজেন্টকে একটি ধারাবাহিক ওয়ার্কফ্লোতে সংযুক্ত করি যা A2A বার্তা প্রেরণের অনুকরণ করে:

1. **CurrencyExchangeAgent** ব্যবহারকারীর অনুরোধ গ্রহণ করে এবং মুদ্রা পরামর্শ তৈরি করে।
2. **ActivityPlannerAgent** সমৃদ্ধ প্রসঙ্গ গ্রহণ করে এবং কার্যক্রমের সুপারিশ যোগ করে।
3. **TravelManagerAgent** উভয় ইনপুটকে একত্রিত করে একটি চূড়ান্ত ভ্রমণ সংক্ষিপ্ত বিবরণ তৈরি করে।


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## উৎপাদন পরিবেশে A2A বোঝা

একটি উৎপাদন পরিবেশে A2A প্রটোকল শক্তিশালী ক্রস-সার্ভিস সিনারিওগুলি উন্মুক্ত করে:

| ক্ষমতা | বিবরণ |
|---|---|
| **বিভিন্ন ফ্রেমওয়ার্কের মধ্যে পারস্পরিক কার্যক্ষমতা** | একটি ফ্রেমওয়ার্ক দিয়ে তৈরি এজেন্ট অন্য কোনও A2A-সমর্থিত ফ্রেমওয়ার্কে তৈরি এজেন্টকে কাজ ধার্য করতে পারে, ফলে প্রকৃত পার-সংগঠন আন্তঃপরিচালন সম্ভব হয়। |
| **সার্ভিস সীমা** | এজেন্টগুলো আলাদা মাইক্রোসার্ভিসে, ক্লাউড রিজিয়নে, বা এমনকি ভিন্ন ভিন্ন সংস্থায় থাকতে পারে, তবুও নির্বিঘ্নে সহযোগিতা করতে সক্ষম। |
| **গতিশীল আবিষ্কার** | একটি অর্কেস্ট্রেটর রানটাইমে Agent Card রেজিস্ট্রিকে অনুসন্ধান করে নির্দিষ্ট সাব-টাস্কের জন্য সবচেয়ে উপযোগী বিশেষজ্ঞ খুঁজে পেতে পারে। |
| **স্ট্রিমিং & পুশ নোটিফিকেশন** | A2A রিয়েল-টাইম অগ্রগতির আপডেটের জন্য Server-Sent Events (SSE) এবং দীর্ঘ সময়চলতি টাস্কগুলির জন্য পুশ নোটিফিকেশন সমর্থন করে। |

The workflow we built above is a simplified, in-process version of this pattern. বাস্তব
ডিপ্লয়মেন্টে প্রতিটি এজেন্ট একটি HTTP endpoint প্রকাশ করবে, একটি Agent Card প্রকাশ করবে, এবং যোগাযোগ
A2A JSON-RPC প্রোটোকলের মাধ্যমে।


## সারাংশ

এই পাঠে আপনি শিখেছেন:

1. **A2A প্রোটোকল কী** — এজেন্ট-টু-এজেন্ট আবিষ্কার, বার্তা আদান-প্রদান,
   এবং কাজ ব্যবস্থাপনা।
2. **কীভাবে বিশেষায়িত এজেন্ট তৈরি করতে হয়** — একটি মুদ্রা বিনিময় এজেন্ট, একটি কার্যকলাপ পরিকল্পনাকারী এজেন্ট, এবং একটি ভ্রমণ ব্যবস্থাপক অর্কেস্ট্রেটর।
3. **কিভাবে এজেন্টগুলোকে একটি ওয়ার্কফ্লোতে সংযুক্ত করতে হয়** — `WorkflowBuilder` ব্যবহার করে ক্রমান্বয়ে
   এজেন্টগুলোর মধ্যে বার্তা প্রেরণের মডেল তৈরি করা।
4. **A2A কীভাবে উৎপাদনে কাজ করে** — ডাইনামিক আবিষ্কার এবং স্ট্রিমিং আপডেটের মাধ্যমে বিভিন্ন ফ্রেমওয়ার্ক ও সার্ভিসের মধ্যে সহযোগিতা সক্ষম করা।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
অস্বীকৃতি:

এই নথিটি AI অনুবাদ সেবা Co-op Translator (https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। আমরা যথাসাধ্য সঠিকতার চেষ্টা করি, তবু অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অযথা অসঙ্গতি থাকতে পারে। মূল নথিটিকে তার নিজ ভাষায় কর্তৃত্বপ্রাপ্ত উৎস হিসেবে বিবেচনা করা উচিত। অত্যন্ত গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ প্রয়োজন। এই অনুবাদ ব্যবহারের ফলে যদি কোনো ভুলবোঝাবুঝি বা ভুল ব্যাখ্যা ঘটে, তাতে আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
